# Uso de LLM con Fine-Tuning en Hugging Face 🧠

---

## 📍¿Qué es el Fine-Tuning y cuándo se utiliza?

El **Fine-Tuning** es el proceso de **ajustar un modelo preentrenado** usando un conjunto de datos específico.  
Esto permite adaptar el conocimiento general del modelo a un dominio concreto, mejorando su rendimiento en tareas personalizadas.

### 🎯 Se utiliza cuando:
- Tienes datos especializados (por ejemplo, textos médicos o legales).  
- Quieres mejorar la precisión de un modelo existente.  
- Necesitas que el modelo hable o razone como un experto en cierto tema.  

---

## 📍Proceso de Fine-Tuning con la librería 🤗 Transformers y Datasets

1️⃣ **Cargar un modelo base** (ej. `distilbert-base-uncased`).  
2️⃣ **Preparar el dataset** (texto y etiquetas).  
3️⃣ **Tokenizar** los datos (convertir texto a números).  
4️⃣ **Entrenar** el modelo con `Trainer` y `TrainingArguments`.  
5️⃣ **Evaluar y guardar** el modelo fine-tuneado.  

# 1️⃣ Cargar un modelo base y un tokenizador

In [1]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model_name = "distilbert-base-uncased"
model = DistilBertForSequenceClassification.from_pretrained(model_name)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# 2️⃣ Dataset de ejemplo en español

In [2]:
from datasets import Dataset

data = {
    "text": [
        # 🌞 Frases positivas
        "Me encanta este producto, funciona de maravilla.",
        "El servicio fue excelente y muy rápido.",
        "Estoy muy feliz con el resultado obtenido.",
        "La atención al cliente fue muy amable y eficiente.",
        "La película fue increíble, me hizo llorar de emoción.",
        "Definitivamente volvería a comprar aquí.",
        "El lugar es hermoso y muy acogedor.",
        "Mi experiencia fue fantástica, todo salió perfecto.",
        "La comida estuvo deliciosa, totalmente recomendada.",
        "Es una de las mejores decisiones que he tomado.",

        # 🌧️ Frases negativas
        "El producto llegó roto y de mala calidad.",
        "La atención fue pésima, no volveré nunca.",
        "Me siento decepcionado, esperaba mucho más.",
        "La película fue aburrida y sin sentido.",
        "El servicio fue lento y los empleados groseros."
    ],
    "label": [
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  # Positivas
        0, 0, 0, 0, 0                   # Negativas
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)
print(dataset[0])

Dataset({
    features: ['text', 'label'],
    num_rows: 15
})
{'text': 'Me encanta este producto, funciona de maravilla.', 'label': 1}


# 3️⃣ Tokenizar los datos

In [3]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

¿Qué es Weights & Biases? https://wandb.ai/site

Weights & Biases (W&B) es una plataforma que ayuda a los equipos de IA/ML a gestionar, monitorear y compartir sus experimentos de forma organizada.

En este caso nos servira para guardar los pesos de nuestro entrenamiento

# 4️⃣ Entrenar el modelo con Trainer y TrainingArguments

In [5]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",          # output directory
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=64,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir="./logs",            # directory for storing logs
    logging_steps=10,
)

trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=tokenized_datasets,    # training dataset
    eval_dataset=tokenized_datasets    # evaluation dataset (optional)
)

trainer.train() # Descomentar para entrenar

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cesardatag5 (cesardatag5-codigog5) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


TrainOutput(global_step=3, training_loss=0.6719820499420166, metrics={'train_runtime': 48.0655, 'train_samples_per_second': 0.936, 'train_steps_per_second': 0.062, 'total_flos': 5961032939520.0, 'train_loss': 0.6719820499420166, 'epoch': 3.0})

In [6]:
# ============================
# GUARDAR EL MODELO ENTRENADO
# ============================
save_directory = "./modelo_bert_finetuned"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"✅ Modelo guardado en: {save_directory}")

✅ Modelo guardado en: ./modelo_bert_finetuned


In [7]:
# ============================
# CARGAR EL MODELO ENTRENADO
# ============================
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "./modelo_bert_finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

In [8]:
# ============================
# FUNCIÓN DE PREDICCIÓN
# ============================
def predecir_texto(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    logits = outputs.logits
    prediccion = torch.argmax(logits, dim=1).item()

    # Si conoces las etiquetas, puedes mapearlas:
    etiquetas = ["negativo", "neutral", "positivo"]  # Ejemplo
    return etiquetas[prediccion] if prediccion < len(etiquetas) else prediccion

# ============================
# PROBAR CON UN TEXTO NUEVO
# ============================
texto_prueba = "La pelicula fue la mejor que vi"
resultado = predecir_texto(texto_prueba)

print(f"Texto: {texto_prueba}")
print(f"Predicción: {resultado}")

Texto: La pelicula fue la mejor que vi
Predicción: neutral
